# 04 – Expansion PM Tiles Map

In [40]:
\
import os
from pathlib import Path
import geopandas as gpd
import pandas as pd

# Use relative paths based on repo structure
REPO_ROOT = Path(__file__).parent.parent if "__file__" in globals() else Path.cwd().parent
OUT_DIR = REPO_ROOT / "docs" / "pmtiles_map"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

BUILDINGS_FILE = DATA_PROCESSED / "buildings_viz.parquet"
FVI_FILE = DATA_RAW / "fvi" / "nyc_flood_vulnerability_index.geojson"
NTA_ZIP = DATA_RAW / "nta" / "nyc_nta_2020.zip"

CRS_NYC = "EPSG:2263"

SCENARIO_LABELS = {
  "ss_cur":  "Storm Surge (Present)",
  "ss_50s":  "Storm Surge (2050s)",
  "ss_80s":  "Storm Surge (2080s)",
  "tid_20s": "Tidal Flooding (2020s)",
  "tid_50s": "Tidal Flooding (2050s)",
  "tid_80s": "Tidal Flooding (2080s)"
}
SCENARIOS = list(SCENARIO_LABELS.keys())

KEEP_BUILDING_COLS = ["objectid", "name", "NTACode","NTAName","boroname","geom_area_sqft","construction_year","geometry"]

print("OUT_DIR:", OUT_DIR)
print("BUILDINGS_FILE exists:", BUILDINGS_FILE.exists())
print("FVI_FILE exists:", FVI_FILE.exists())

OUT_DIR: c:\Code\NYCDSA\nyc-urban-built-floodzone-planimetrics\docs\pmtiles_map
BUILDINGS_FILE exists: True
FVI_FILE exists: True


## Load files

In [41]:
\
buildings = gpd.read_parquet(BUILDINGS_FILE).set_crs(CRS_NYC, allow_override=True)
fvi = gpd.read_file(FVI_FILE).to_crs(CRS_NYC)

# FVI Review
print("FVI columns:", list(fvi.columns))
print("FVI CRS:", fvi.crs)
print("FVI shape:", fvi.shape)
print(fvi.head())

# Bulilding review
print("Buildings columns:", list(buildings.columns))
print("Buildings CRS:", buildings.crs)
print("Buildings shape:", buildings.shape)
print(buildings.head())

FVI columns: [':id', ':version', ':created_at', ':updated_at', 'geoid', 'fshri', 'ss_cur', 'ss_50s', 'ss_80s', 'tid_20s', 'tid_50s', 'tid_80s', 'geometry']
FVI CRS: EPSG:2263
FVI shape: (2209, 13)
                  :id           :version                      :created_at  \
0  row-b9ez~tyyc.hzqp  rv-pm7b-b3hv-p8yb 2024-03-06 19:39:30.190000+00:00   
1  row-uy6c-pzz4.87a2  rv-yumb-mbxz_tt73 2024-03-06 19:39:30.190000+00:00   
2  row-w7q9.pdp5~istm  rv-kvy3-v544-uupv 2024-03-06 19:39:30.190000+00:00   
3  row-ir59~neg3-fng2  rv-gawd-q4zc~vyhw 2024-03-06 19:39:30.190000+00:00   
4  row-htt9_yg3g.emcg  rv-c8dc-pcmg.y87e 2024-03-06 19:39:30.190000+00:00   

                       :updated_at        geoid fshri ss_cur ss_50s ss_80s  \
0 2024-03-06 19:39:30.190000+00:00  36005008500     5   None   None   None   
1 2024-03-06 19:39:30.190000+00:00  36081008600     5   None   None   None   
2 2024-03-06 19:39:30.190000+00:00  36047058000     5   None   None      2   
3 2024-03-06 19:39:30.190000

## Join FVI (flood risk) to each building, spaital join

In [42]:
bld_pts = buildings[["objectid", "geometry"]].copy()
bld_pts["geometry"] = bld_pts.geometry.centroid

fvi_zones = fvi[["geometry"] + SCENARIOS].copy()

#compare bounds
print("#Bounds should be similiar to fit fitting NYC#")
print("Building centroids bounds:", bld_pts.total_bounds)
print("FVI zones bounds:", fvi_zones.total_bounds)

bld_fvi = gpd.sjoin(bld_pts, fvi_zones, how="left", predicate="within").drop(columns=["index_right"], errors="ignore")
print("Review join: ")
print("bld_fvi rows:", len(bld_fvi))
print("Buildings an FVI Score: ", 
      bld_fvi[SCENARIOS].notna().any(axis=1).sum())
print("Buildings with all null FVI values:", 
      bld_fvi[SCENARIOS].isna().all(axis=1).sum())

# Review matched vs unmatched
print("Show matched buildings:")
matched = bld_fvi[bld_fvi[SCENARIOS].notna().any(axis=1)]
print(matched.head())


print("Show unmatched buildings:")
unmatched = bld_fvi[bld_fvi[SCENARIOS].isna().all(axis=1)]
print(unmatched.head())

bld_fvi.head()

#Bounds should be similiar to fit fitting NYC#
Building centroids bounds: [ 913261.19916494  120980.93213421 1067304.77277139  272635.42264042]
FVI zones bounds: [ 913164.86204405  120117.02334761 1067283.13044668  272608.58232303]
Review join: 
bld_fvi rows: 1082999
Buildings an FVI Score:  323334
Buildings with all null FVI values: 759665
Show matched buildings:
    objectid                        geometry ss_cur ss_50s ss_80s tid_20s  \
5   246429.0   POINT (952434.647 148591.889)      1      1      2    None   
11  633468.0  POINT (1027435.044 176531.532)      4      4      4    None   
12  374149.0   POINT (998537.232 159161.528)   None      2      4    None   
17  548372.0   POINT (945184.248 140695.823)      1      1      1    None   
21  607770.0  POINT (1013975.456 175115.017)      4      5      5    None   

   tid_50s tid_80s  
5     None    None  
11    None       3  
12    None    None  
17    None    None  
21    None       3  
Show unmatched buildings:
   objectid       

,objectid,geometry,ss_cur,ss_50s,ss_80s,tid_20s,tid_50s,tid_80s
0,507357.0,POINT (1052386.895 214157.488),None,None,None,None,None,None
1,137879.0,POINT (1052753.726 201306.083),None,None,None,None,None,None
2,982953.0,POINT (988739.603 159096.157),None,None,None,None,None,None
3,244121.0,POINT (1012449.66 181747.497),None,None,None,None,None,None
4,229537.0,POINT (1020192.208 267285.288),None,None,None,None,None,None


## Aggregate to building level score per scenario (max severity)

In [43]:
\
bld_fvi_agg = bld_fvi.groupby("objectid", dropna=False)[SCENARIOS].max().reset_index()
use_cols = [c for c in KEEP_BUILDING_COLS if c in buildings.columns]
bld_poly = buildings[use_cols].copy().merge(bld_fvi_agg, on="objectid", how="left")
print("bld_poly:", len(bld_poly), "cols:", len(bld_poly.columns))
bld_poly.head()

bld_poly: 1082999 cols: 14


,objectid,name,NTACode,NTAName,boroname,geom_area_sqft,construction_year,geometry,ss_cur,ss_50s,ss_80s,tid_20s,tid_50s,tid_80s
0,507357.0,None,QN1102,Bayside,Queens,1096.759439,1950,"POLYGON ((1052359.953 214149.188, 1052400.668 ...",NaN,NaN,NaN,NaN,NaN,NaN
1,137879.0,None,QN1303,Queens Village,Queens,214.631781,1930,"POLYGON ((1052764.922 201306.381, 1052748.182 ...",NaN,NaN,NaN,NaN,NaN,NaN
2,982953.0,None,BK1103,Gravesend (West),Brooklyn,1121.202926,1915,"POLYGON ((988769.316 159090.345, 988746.041 15...",NaN,NaN,NaN,NaN,NaN,NaN
3,244121.0,None,BK0503,East New York-New Lots,Brooklyn,656.015069,1997,"POLYGON ((1012469.24 181742.107, 1012446.631 1...",NaN,NaN,NaN,NaN,NaN,NaN
4,229537.0,None,BX1203,Wakefield-Woodlawn,Bronx,1334.183950,1910,"POLYGON ((1020199.824 267310.273, 1020201.035 ...",NaN,NaN,NaN,NaN,NaN,NaN


## Export to geojson

In [44]:
\
bld_poly["geometry"] = bld_poly.geometry.simplify(2.0, preserve_topology=True)
bld_wgs84 = bld_poly.to_crs(4326)

buildings_geojson = OUT_DIR / "buildings_wgs84_fvi.geojson"
bld_wgs84.to_file(buildings_geojson, driver="GeoJSON")
print("Saved:", buildings_geojson, "features:", len(bld_wgs84))

Saved: c:\Code\NYCDSA\nyc-urban-built-floodzone-planimetrics\docs\pmtiles_map\buildings_wgs84_fvi.geojson features: 1082999


## Export floodzones geojson

In [45]:
\
scenario_gdfs = []
for scenario in SCENARIOS:
    tmp = fvi[["geometry", scenario]].copy()
    tmp = tmp[tmp[scenario].notna()]
    if len(tmp) == 0:
        continue
    tmp = tmp.dissolve()
    tmp["scenario"] = scenario
    scenario_gdfs.append(tmp[["scenario", "geometry"]])

floodzones = gpd.GeoDataFrame(pd.concat(scenario_gdfs, ignore_index=True), crs=CRS_NYC).to_crs(4326)

floodzones_geojson = OUT_DIR / "floodzones_wgs84.geojson"
floodzones.to_file(floodzones_geojson, driver="GeoJSON")
print("Saved:", floodzones_geojson, "features:")

Saved: c:\Code\NYCDSA\nyc-urban-built-floodzone-planimetrics\docs\pmtiles_map\floodzones_wgs84.geojson features:


## Export NTA

In [46]:
if NTA_ZIP.exists():
    nta = gpd.read_file(f"zip://{NTA_ZIP}")
    
    nta = nta.rename(columns={
        "nta2020": "NTACode",
        "ntaname": "NTAName"
    })
    
    nta = nta[["NTACode", "NTAName", "boroname", "geometry"]].to_crs(CRS_NYC).to_crs(4326)
    
    # simplifies geometry for performance
    nta["geometry"] = nta.geometry.simplify(0.0001, preserve_topology=True)
    
   
    nta_geojson = OUT_DIR / "nta_wgs84.geojson"
    nta.to_file(nta_geojson, driver="GeoJSON")
    print("Saved:", nta_geojson)


Saved: c:\Code\NYCDSA\nyc-urban-built-floodzone-planimetrics\docs\pmtiles_map\nta_wgs84.geojson


## KPI Summaries

In [47]:
df = pd.DataFrame(bld_poly.drop(columns="geometry"))
df["geom_area_sqft"] = pd.to_numeric(df.get("geom_area_sqft"), errors="coerce")

kpis = {}
for s in SCENARIOS:
    v = pd.to_numeric(df.get(s), errors="coerce")
    flooded = v.fillna(0) >= 1

    total_area = df.loc[flooded, "geom_area_sqft"].sum(skipna=True)
    total_count = int(flooded.sum())

    area_by = {}
    count_by = {}
    for lvl in [1,2,3,4,5]:
        m = v == lvl
        area_by[str(lvl)] = float(df.loc[m, "geom_area_sqft"].sum(skipna=True) or 0.0)
        count_by[str(lvl)] = int(m.sum())

    kpis[s] = {"total_area": float(total_area or 0.0), "total_count": total_count, "area_by": area_by, "count_by": count_by}

kpis["__labels__"] = SCENARIO_LABELS
print("KPI scenarios:", [k for k in kpis.keys() if not k.startswith("__")])

# Export KPI data as JSON for potential dashboarding visualization
import json
kpi_json_file = OUT_DIR / "kpi_data.json"
with open(kpi_json_file, "w", encoding="utf-8") as f:
    json.dump(kpis, f, indent=2)
print("Saved:", kpi_json_file)

KPI scenarios: ['ss_cur', 'ss_50s', 'ss_80s', 'tid_20s', 'tid_50s', 'tid_80s']
Saved: c:\Code\NYCDSA\nyc-urban-built-floodzone-planimetrics\docs\pmtiles_map\kpi_data.json
